# Run an Everruns agent from a notebook

This notebook creates a small agent with the Python SDK, starts a session on the **Generic** harness, sends one message, and streams the reply inline.

It defaults to `https://app.everruns.com/api`. Override `EVERRUNS_API_URL` only when you want to point it at a local or self-hosted Everruns instance.

## Before you run it

- Set `EVERRUNS_API_KEY` when you run against `app.everruns.com`
- Make sure your org has a usable default model configured
- CI uses the same notebook but overrides `EVERRUNS_API_URL` to a local dev-mode server and turns on `EVERRUNS_NOTEBOOK_USE_LLMSIM=1`

In [1]:
import os
import subprocess
import sys

if os.environ.get("EVERRUNS_NOTEBOOK_SKIP_PIP") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "everruns-sdk", "requests"])

In [2]:
import asyncio
import os
import uuid

import requests
from everruns_sdk import Everruns

BASE_URL = os.environ.get("EVERRUNS_API_URL", "https://app.everruns.com/api")
API_KEY = os.environ.get("EVERRUNS_API_KEY", "")
USE_LLMSIM = os.environ.get("EVERRUNS_NOTEBOOK_USE_LLMSIM") == "1"

if BASE_URL.startswith("https://app.everruns.com") and not API_KEY:
    raise RuntimeError("Set EVERRUNS_API_KEY before running this notebook against app.everruns.com.")

if not API_KEY:
    API_KEY = "dev"

client = Everruns(api_key=API_KEY, base_url=BASE_URL)
headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

In [3]:
def ensure_ci_llmsim_model():
    if not USE_LLMSIM:
        return None

    suffix = uuid.uuid4().hex[:8]
    provider = requests.post(
        f"{BASE_URL}/v1/llm-providers",
        headers=headers,
        json={"name": f"Notebook LlmSim {suffix}", "provider_type": "llmsim"},
        timeout=30,
    )
    provider.raise_for_status()
    provider_id = provider.json()["id"]

    model = requests.post(
        f"{BASE_URL}/v1/llm-providers/{provider_id}/models",
        headers=headers,
        json={
            "model_id": f"llmsim-notebook-{suffix}",
            "display_name": "Notebook LlmSim",
        },
        timeout=30,
    )
    model.raise_for_status()
    return model.json()["id"]


def extract_text(message):
    parts = []
    for part in message.get("content", []):
        if part.get("type") == "text":
            parts.append(part["text"])
    return "\n".join(parts).strip()

In [4]:
async def run_demo():
    model_id = ensure_ci_llmsim_model()

    agent = await client.agents.create(
        name="notebook-quickstart",
        system_prompt=(
            "You are a concise demo assistant. "
            "Answer in short bullet points and mention durability when relevant."
        ),
        default_model_id=model_id,
    )

    session = await client.sessions.create(
        harness_name="generic",
        agent_id=agent.id,
        title="Notebook quickstart",
    )

    await client.messages.create(
        session.id,
        "What does Everruns do, and why would a product team use it?",
    )

    streamed_chunks = []
    completed_text = None

    async for event in client.events.stream(session.id):
        if event.type == "output.message.delta":
            delta = event.data.get("delta", "")
            streamed_chunks.append(delta)
            print(delta, end="", flush=True)
        elif event.type == "output.message.completed":
            completed_text = extract_text(event.data.get("message", {}))
        elif event.type == "turn.completed":
            break
        elif event.type == "turn.failed":
            raise RuntimeError(event.data.get("error", "turn failed"))

    response_text = "".join(streamed_chunks).strip() or completed_text or ""
    return {
        "base_url": BASE_URL,
        "agent_id": agent.id,
        "session_id": session.id,
        "response": response_text,
    }


result = await run_demo()
result

In [5]:
result

- Everruns runs durable agent workflows behind a stable API and session model.
- It keeps agent state, tool execution, and event streams together so retries stay predictable.
- Product teams use it when they want streaming, durable agents without rebuilding the control plane themselves.


{'base_url': 'https://app.everruns.com/api',
 'agent_id': 'agent_0196831a1f9c7a0f8a5c9ab3d7d0f201',
 'session_id': 'session_0196831a20837ddbb05fd8f6d0c8f64a',
 'response': '- Everruns runs durable agent workflows behind a stable API and session model.\n- It keeps agent state, tool execution, and event streams together so retries stay predictable.\n- Product teams use it when they want streaming, durable agents without rebuilding the control plane themselves.'}

## Next steps

- Change the prompt and rerun the final cells
- Point `EVERRUNS_API_URL` at a local or self-hosted Everruns deployment when you want to test outside `app.everruns.com`
- Move the same flow into a script or service once the notebook feels right